In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("simonbouchardk/book-recommendation-platform-data")

print("Path to dataset files:", path)

100%|██████████| 48.9M/48.9M [00:02<00:00, 22.0MB/s]

Extracting files...


Path to dataset files: /home/mysa/.cache/kagglehub/datasets/simonbouchardk/book-recommendation-platform-data/versions/1


In [6]:
import pandas as pd

books = pd.read_csv(f"{path}/books.csv")
book_tags = pd.read_csv(f"{path}/books_to_subjects.csv")

In [4]:
books

,work_id,filled_work_id,title,year,year_bucket,filled_year,description,cover_id,language,num_pages,filled_num_pages,author_id,isbn
0,OL1888129W,False,Classical mythology,2002,early_2000s,False,An excellent primer on classical mythology for...,5461546,eng,524,False,OL226080A,0582280044
1,OL5109697W,False,Clara Callan,2001,early_2000s,False,"E-book extras: ""Hero of the Humdrum"": A profil...",26064,eng,530,False,OL1125685A,000639406X
2,OL2959148W,False,Decision in Normandy,1991,late_20th,False,No description available,811606,eng,753,False,OL451959A,0060973129
3,OL2699188W,False,Flu,1999,late_20th,False,"The fascinating, true story of the world's dea...",6968144,eng,330,False,OL2632232A,0743203984
4,OL1879848W,False,The mummies of Ürümchi,1999,late_20th,False,"In the museums of Urumchi, the wind-swept regi...",10708239,eng,240,False,OL225040A,0393045218
...,...,...,...,...,...,...,...,...,...,...,...,...,...
251607,OL2223342W,False,Friends and Kangaroos (New Writing Scotland),1999,late_20th,False,No description available,11348096,eng,164,False,OL285948A,0948877413
251608,OL12764222W,False,My hard heart,1998,late_20th,False,No description available,-1,Unknown,251,True,OL335680A,067087874X
251609,OL1242837W,False,Touriste de bananes,1938,classic,False,No description available,-1,eng,212,False,OL125391A,8845912264
251610,OL4225425W,False,Price of Happiness,1977,mid_20th,False,**It was a simple lie\r\nBut he doubted her**\...,13180399,eng,192,False,OL808883A,0263091538


- all columns report 0 None count but None values are sometimes just hidden
- 5127 books have year 0 (likely just missing)
- 73% of books have "No description available" in the description field
- 82% of books are in english -- drop other languages
- num_pages also contains values that do not make sense (e.g. 980227), most of the books have this missing

In [7]:
book_tags

,work_id,subjects
0,OL1888129W,Mythology
1,OL1888129W,Classical Mythology
2,OL1888129W,Classical
3,OL5109697W,Ontario
4,OL5109697W,Psychological Fiction
...,...,...
912522,OL1242837W,[NO_SUBJECT]
912523,OL4225425W,[NO_SUBJECT]
912524,OL5094308W,Sound
912525,OL5094308W,Fiction


#### Remove non-english books

In [8]:
from src.catalog.normalize import is_english
books = books[is_english(books["language"], keep_missing=True)]
books

,work_id,filled_work_id,title,year,year_bucket,filled_year,description,cover_id,language,num_pages,filled_num_pages,author_id,isbn
0,OL1888129W,False,Classical mythology,2002,early_2000s,False,An excellent primer on classical mythology for...,5461546,eng,524,False,OL226080A,0582280044
1,OL5109697W,False,Clara Callan,2001,early_2000s,False,"E-book extras: ""Hero of the Humdrum"": A profil...",26064,eng,530,False,OL1125685A,000639406X
2,OL2959148W,False,Decision in Normandy,1991,late_20th,False,No description available,811606,eng,753,False,OL451959A,0060973129
3,OL2699188W,False,Flu,1999,late_20th,False,"The fascinating, true story of the world's dea...",6968144,eng,330,False,OL2632232A,0743203984
4,OL1879848W,False,The mummies of Ürümchi,1999,late_20th,False,"In the museums of Urumchi, the wind-swept regi...",10708239,eng,240,False,OL225040A,0393045218
...,...,...,...,...,...,...,...,...,...,...,...,...,...
251607,OL2223342W,False,Friends and Kangaroos (New Writing Scotland),1999,late_20th,False,No description available,11348096,eng,164,False,OL285948A,0948877413
251608,OL12764222W,False,My hard heart,1998,late_20th,False,No description available,-1,Unknown,251,True,OL335680A,067087874X
251609,OL1242837W,False,Touriste de bananes,1938,classic,False,No description available,-1,eng,212,False,OL125391A,8845912264
251610,OL4225425W,False,Price of Happiness,1977,mid_20th,False,**It was a simple lie\r\nBut he doubted her**\...,13180399,eng,192,False,OL808883A,0263091538


#### Remove books with no description

In [9]:
from src.catalog.normalize import replace_placeholders, BOOKCROSSING_BOOKS_PLACEHOLDERS
books = replace_placeholders(books, BOOKCROSSING_BOOKS_PLACEHOLDERS, ["description"])

,work_id,filled_work_id,title,year,year_bucket,filled_year,description,cover_id,language,num_pages,filled_num_pages,author_id,isbn
0,OL1888129W,False,Classical mythology,2002,early_2000s,False,An excellent primer on classical mythology for...,5461546,eng,524,False,OL226080A,0582280044
1,OL5109697W,False,Clara Callan,2001,early_2000s,False,"E-book extras: ""Hero of the Humdrum"": A profil...",26064,eng,530,False,OL1125685A,000639406X
2,OL2959148W,False,Decision in Normandy,1991,late_20th,False,NaN,811606,eng,753,False,OL451959A,0060973129
3,OL2699188W,False,Flu,1999,late_20th,False,"The fascinating, true story of the world's dea...",6968144,eng,330,False,OL2632232A,0743203984
4,OL1879848W,False,The mummies of Ürümchi,1999,late_20th,False,"In the museums of Urumchi, the wind-swept regi...",10708239,eng,240,False,OL225040A,0393045218
...,...,...,...,...,...,...,...,...,...,...,...,...,...
251607,OL2223342W,False,Friends and Kangaroos (New Writing Scotland),1999,late_20th,False,NaN,11348096,eng,164,False,OL285948A,0948877413
251608,OL12764222W,False,My hard heart,1998,late_20th,False,NaN,-1,Unknown,251,True,OL335680A,067087874X
251609,OL1242837W,False,Touriste de bananes,1938,classic,False,NaN,-1,eng,212,False,OL125391A,8845912264
251610,OL4225425W,False,Price of Happiness,1977,mid_20th,False,**It was a simple lie\r\nBut he doubted her**\...,13180399,eng,192,False,OL808883A,0263091538


In [1]:
import kagglehub
path = kagglehub.dataset_download("simonbouchardk/books-crossing-processed-full-version")

/home/mysa/PycharmProjects/book-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 86.9M/86.9M [00:04<00:00, 21.0MB/s]

Extracting files...
